In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from matplotlib.lines import Line2D

sys.path.append("/workspace")

from vertexcbf.config_utils import build_constr_fn, build_dynamics, build_model
from vertexcbf.validation import validate_cbf

%reload_ext autoreload
%autoreload 2

WORKSPACE = "/workspace"
DEVICE = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

# Methods to compare, in left-to-right order.  Each tuple is
# (checkpoint sub-folder under checkpoints/, column title).  Edit this
# list to add/remove methods or reorder columns. Missing checkpoints are
# skipped — the column stays in place but is left blank.
METHODS = [
    ("NO_DATA",  "PDE-only"),
    ("FC_DATA",  "FCD"),
    ("VRC_DATA", "VRCD (ours)"),
]

# DPI for the rasterized heatmap layer in saved PDFs. Contours, text, axes
# and colorbar stay vector regardless.
SAVE_DPI = 200

Path("figures/compare").mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE}")
print(f"Methods: {[m[0] for m in METHODS]}")


def _load_system(system, method):
    """Load config, dynamics, model, ground truth for one (system, method) pair.

    Returns None if no checkpoint exists for this (method, system).
    """
    cfg_path = f"{WORKSPACE}/configs/{system}.yaml"
    ckpt_path = f"{WORKSPACE}/checkpoints/{method}/{system}/final.pt"
    truth_dir = f"{WORKSPACE}/data/true_values/{system}"

    if not os.path.isfile(ckpt_path):
        return None

    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    dynamics = build_dynamics(cfg["system"], device=DEVICE)
    constr_fn = build_constr_fn(cfg["constraint"])
    model = build_model(cfg["model"], dynamics, device=DEVICE)

    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    values_true = grid_true = None
    if os.path.isfile(f"{truth_dir}/values.npy"):
        values_true = np.load(f"{truth_dir}/values.npy")
        grid_true = np.load(f"{truth_dir}/grid.npy")

    return cfg, dynamics, constr_fn, model, values_true, grid_true


def _grid_axis_coords(grid_true):
    nx = grid_true.shape[-1]
    coords = []
    for i in range(nx):
        sel = [0] * nx
        sel[i] = slice(None)
        coords.append(grid_true[tuple(sel) + (i,)])
    return coords


def _slice_ground_truth(values_true, grid_true, slice_axes, slice_fixed_values):
    nx = grid_true.shape[-1]
    if grid_true.ndim == 3:
        return grid_true[..., slice_axes[0]], grid_true[..., slice_axes[1]], values_true

    coords = _grid_axis_coords(grid_true)
    lo, hi = sorted(slice_axes)
    indexer = [None] * nx
    indexer[lo] = slice(None)
    indexer[hi] = slice(None)
    for i in range(nx):
        if i in slice_axes:
            continue
        if i not in slice_fixed_values:
            return None
        indexer[i] = int(np.argmin(np.abs(coords[i] - slice_fixed_values[i])))

    values_2d = values_true[tuple(indexer)]
    if slice_axes != (lo, hi):
        values_2d = values_2d.T

    tg0, tg1 = np.meshgrid(coords[slice_axes[0]], coords[slice_axes[1]], indexing="ij")
    return tg0, tg1, values_2d


def _build_slice(dynamics, slice_axes, slice_fixed_values, slice_shape, grid_true):
    ax0, ax1 = slice_axes
    other_axes = [i for i in range(dynamics.nx) if i not in slice_axes]

    if grid_true is not None and grid_true.ndim == 3:
        g0 = grid_true[..., ax0]
        g1 = grid_true[..., ax1]
        slice_states = grid_true.reshape(-1, dynamics.nx).astype(np.float32)
        return g0, g1, slice_states

    missing = [i for i in other_axes if i not in slice_fixed_values]
    if missing:
        raise ValueError(f"slice_fixed_values must specify axes {missing}")

    x_min = dynamics.x_min.cpu().numpy()
    x_max = dynamics.x_max.cpu().numpy()
    n0, n1 = slice_shape
    g0, g1 = np.meshgrid(
        np.linspace(x_min[ax0], x_max[ax0], n0),
        np.linspace(x_min[ax1], x_max[ax1], n1),
        indexing="ij",
    )

    slice_states = np.zeros((n0 * n1, dynamics.nx), dtype=np.float32)
    slice_states[:, ax0] = g0.ravel()
    slice_states[:, ax1] = g1.ravel()
    for i in other_axes:
        slice_states[:, i] = slice_fixed_values[i]
    return g0, g1, slice_states


def _predict_values(model, constr_fn, states_np):
    with torch.no_grad():
        x = torch.as_tensor(states_np, dtype=torch.float32, device=DEVICE)
        return (constr_fn(x) - model(x)).squeeze(-1).cpu().numpy()


def _compute_method(system, method, slice_axes, slice_fixed_values, slice_shape, include_validated):
    """Run one method end-to-end on the requested 2-D slice. Returns None if no checkpoint."""
    loaded = _load_system(system, method)
    if loaded is None:
        return None
    cfg, dynamics, constr_fn, model, values_true, grid_true = loaded
    g0, g1, slice_states = _build_slice(
        dynamics, slice_axes, slice_fixed_values, slice_shape, grid_true,
    )
    n0, n1 = g0.shape
    values_pred = _predict_values(model, constr_fn, slice_states).reshape(n0, n1)
    states_t = torch.as_tensor(slice_states, dtype=torch.float32, device=DEVICE)
    constr = constr_fn(states_t).cpu().numpy().reshape(n0, n1)

    values_val = None
    if include_validated:
        val_cfg = cfg.get("validation", {}) or {}
        slice_val = validate_cbf(
            dynamics=dynamics,
            states=states_t,
            constr_fn=constr_fn,
            residual_fn=model,
            T=val_cfg["T"],
            dt=val_cfg["dt"],
            return_values=True,
            batch_size=val_cfg.get("batch_size"),
        )
        values_val = slice_val["values"].cpu().numpy().reshape(n0, n1)

    gt_slice = None
    if values_true is not None:
        gt_slice = _slice_ground_truth(
            values_true, grid_true, slice_axes, slice_fixed_values,
        )

    return {
        "dynamics": dynamics,
        "g0": g0,
        "g1": g1,
        "values_pred": values_pred,
        "values_val": values_val,
        "constr": constr,
        "gt_slice": gt_slice,
    }


def compare_methods(
    system,
    slice_axes,
    slice_axes_labels,
    slice_fixed_values=None,
    slice_shape=(200, 200),
    include_validated=True,
    figsize=None,
    save=True,
):
    """1xN comparison of V(x) across METHODS on the same 2-D slice.

    All subplots share one colorbar (right of figure) using a global vmin/vmax
    computed across the methods that have checkpoints. Methods without a
    checkpoint keep their column slot but stay blank.
    """
    slice_fixed_values = slice_fixed_values or {}
    results = []
    for method, title in METHODS:
        r = _compute_method(
            system, method, slice_axes, slice_fixed_values, slice_shape, include_validated,
        )
        if r is not None:
            r["method"] = method
            r["title"] = title
        else:
            print(f"  [skip] no checkpoint for {method}/{system}")
        results.append((method, title, r))

    available = [r for _, _, r in results if r is not None]
    if not available:
        raise FileNotFoundError(f"No checkpoints found for {system} in any of {[m for m, _ in METHODS]}")

    # One colormap normalization shared by every populated subplot.
    vmin = float(min(r["values_pred"].min() for r in available))
    vmax = float(max(r["values_pred"].max() for r in available))

    dynamics = available[0]["dynamics"]
    ax0, ax1 = slice_axes
    x_min = dynamics.x_min.cpu().numpy()
    x_max = dynamics.x_max.cpu().numpy()

    n_methods = len(results)
    if figsize is None:
        figsize = (5 * n_methods, 5)

    fig, axes = plt.subplots(
        1, n_methods, figsize=figsize, sharey=True, constrained_layout=True,
    )
    if n_methods == 1:
        axes = [axes]

    im = None
    for ax, (method, title, r) in zip(axes, results):
        ax.set_title(title, fontsize=16)
        ax.set_xlabel(slice_axes_labels[0], fontsize=14)
        ax.set_xlim(x_min[ax0], x_max[ax0])
        ax.set_ylim(x_min[ax1], x_max[ax1])

        if r is None:
            # Leave column blank but keep the slot so other columns stay aligned.
            ax.text(
                0.5, 0.5, "(not available)",
                transform=ax.transAxes, ha="center", va="center",
                fontsize=10, color="gray",
            )
            continue

        im = ax.pcolormesh(
            r["g0"], r["g1"], r["values_pred"],
            cmap="plasma", shading="auto", rasterized=True,
            vmin=vmin, vmax=vmax,
        )
        ax.contour(r["g0"], r["g1"], r["constr"], levels=[0], colors="black", linewidths=2.0)

        if r["gt_slice"] is not None:
            tg0, tg1, values_true_2d = r["gt_slice"]
            ax.contour(tg0, tg1, values_true_2d, levels=[0], colors="yellow", linewidths=1.5)

        ax.contour(
            r["g0"], r["g1"], r["values_pred"], levels=[0],
            colors="dodgerblue", linewidths=1.5, linestyles="--",
        )

        if r["values_val"] is not None:
            ax.contour(
                r["g0"], r["g1"], r["values_val"], levels=[0],
                colors="lime", linewidths=1.5, linestyles="-.",
            )

    axes[0].set_ylabel(slice_axes_labels[1], fontsize=14)

    # Shared legend on the first populated axes.
    handles = [Line2D([0], [0], color="black", lw=2.0, label="Constraint")]
    if any(r["gt_slice"] is not None for r in available):
        handles.append(Line2D([0], [0], color="yellow", lw=1.5, label="Ground truth"))
    handles.append(Line2D([0], [0], color="dodgerblue", lw=1.5, ls="--", label="Predicted"))
    if include_validated:
        handles.append(Line2D([0], [0], color="lime", lw=1.5, ls="-.", label="Validated"))
    first_populated = next(ax for ax, (_, _, r) in zip(axes, results) if r is not None)
    first_populated.legend(handles=handles, loc="best", fontsize=11)

    # Single colorbar to the right of the whole figure.
    cbar = fig.colorbar(im, ax=axes, location="right", shrink=0.95, pad=0.02)
    cbar.ax.set_title(r"$V_{\Theta}(x)$", pad=10, fontsize=14)

    if save:
        fig.savefig(
            f"figures/compare/compare_{system}.pdf",
            bbox_inches="tight",
            dpi=SAVE_DPI,
        )

In [ ]:
compare_methods(
    system="inverted_pendulum",
    slice_axes=(0, 1),
    slice_axes_labels=(r"$\theta$", r"$\omega$"),
    include_validated=False,
)

In [ ]:
# =====================================================================
# Trajectory comparison: visualize state trajectories produced by
# different trajectory-optimization methods on the same 2-D slice.
#
# Edit ``TRAJ_METHODS`` below to add/remove methods or tweak their
# hyperparameters. Each entry is a dict:
#   - "title":  column title in the figure
#   - "fn":     the method callable (must accept ``return_trajs=True``
#               and return ``state_traj`` of shape (N, K+1, nx))
#   - "kwargs": method-specific kwargs (budget, horizon, dt, extras).
#               Beam-family uses ``B`` (beam width); shooting uses ``N_s``.
# =====================================================================

from vertexcbf.trajopt import (
    beam_search,
    stochastic_beam_search,
    branch_and_bound,
    mppi,
)

TRAJ_METHODS = [
    {
        "title": "full-control MPPI",
        "fn": mppi,
        "kwargs": dict(N_s=500, K=20, dt=0.1, sigma=1.225, lam=0.0, n_iter=5),
    },
    {
        "title": "vertex-restricted BS (ours)",
        "fn": beam_search,
        "kwargs": dict(B=500, K=20, dt=0.1),
    },
]


def _run_method_trajs(spec, dynamics, constr_fn, x0):
    """Run one method with ``return_trajs=True`` and return (N, K+1, nx) numpy."""
    out = spec["fn"](
        dynamics=dynamics,
        x0=x0,
        constr_fn=constr_fn,
        return_trajs=True,
        **spec["kwargs"],
    )
    return out["state_traj"].detach().cpu().numpy()


def _tail_min(c_traj):
    """Cumulative min over the suffix k' >= k along axis=1. ``c_traj`` shape (N, K+1)."""
    return np.flip(
        np.minimum.accumulate(np.flip(c_traj, axis=1), axis=1), axis=1
    )


def compare_trajectories(
    system,
    methods,
    slice_axes,
    slice_axes_labels,
    n_init=8,
    seed=0,
    slice_fixed_values=None,
    constr_grid_shape=(200, 200),
    sample_within_safe_only=False,
    figsize=None,
    save=True,
):
    """1xN comparison of state trajectories under different trajectory-optimization methods.

    Each panel shows:
      * the ``c(x) = 0`` contour (black) on a clean white background;
      * ``n_init`` random initial states, drawn as large filled circles
        coloured by ``min_{k >= 0} c(x_k)`` (the worst constraint value
        reached along the trajectory);
      * the subsequent trajectory states as small semi-transparent black
        dots, connected by thin grey lines.

    A single colorbar (shared across panels) shows the initial-marker
    colour scale.

    Args:
        system:                 Config name under ``configs/`` (no extension).
        methods:                List of method specs (see ``TRAJ_METHODS``).
        slice_axes:             (i, j) state indices spanning the plot.
        slice_axes_labels:      (xlabel, ylabel) for the plot.
        n_init:                 Number of random initial states.
        seed:                   RNG seed for reproducibility of initial states.
        slice_fixed_values:     For systems with nx > 2, fixes the off-slice
                                state components to these values.
        constr_grid_shape:      Grid resolution for the constraint contour.
        sample_within_safe_only: If True, reject initial states outside
                                ``{c(x) > 0}``.
        figsize, save:          Standard.

    Returns:
        The matplotlib Figure.
    """
    slice_fixed_values = slice_fixed_values or {}

    # Load system: only dynamics + constraint are needed (no learned model).
    cfg_path = f"{WORKSPACE}/configs/{system}.yaml"
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    dynamics = build_dynamics(cfg["system"], device=DEVICE)
    constr_fn = build_constr_fn(cfg["constraint"])

    ax0, ax1 = slice_axes
    x_min = dynamics.x_min.cpu().numpy()
    x_max = dynamics.x_max.cpu().numpy()

    # ------------------------------------------------------------------
    # Random initial states on the slice.
    # ------------------------------------------------------------------
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    if sample_within_safe_only:
        # Rejection sampling until we collect ``n_init`` states with c(x) > 0.
        buf = []
        attempts = 0
        while sum(b.shape[0] for b in buf) < n_init and attempts < 100:
            cand = (
                torch.rand(4 * n_init, dynamics.nx, generator=g, device=DEVICE)
                * (dynamics.x_max - dynamics.x_min)
                + dynamics.x_min
            )
            for i, v in slice_fixed_values.items():
                cand[:, i] = v
            with torch.no_grad():
                cv = constr_fn(cand).reshape(-1)
            buf.append(cand[cv > 0])
            attempts += 1
        x0 = torch.cat(buf, dim=0)[:n_init]
    else:
        x0 = (
            torch.rand(n_init, dynamics.nx, generator=g, device=DEVICE)
            * (dynamics.x_max - dynamics.x_min)
            + dynamics.x_min
        )
        for i, v in slice_fixed_values.items():
            x0[:, i] = v

    # ------------------------------------------------------------------
    # Constraint grid on the slice (used to draw the c=0 contour only).
    # ------------------------------------------------------------------
    n0, n1 = constr_grid_shape
    g0, g1 = np.meshgrid(
        np.linspace(x_min[ax0], x_max[ax0], n0),
        np.linspace(x_min[ax1], x_max[ax1], n1),
        indexing="ij",
    )
    slice_states = np.zeros((n0 * n1, dynamics.nx), dtype=np.float32)
    slice_states[:, ax0] = g0.ravel()
    slice_states[:, ax1] = g1.ravel()
    for i in range(dynamics.nx):
        if i not in slice_axes:
            slice_states[:, i] = slice_fixed_values.get(i, 0.0)
    with torch.no_grad():
        constr_grid = (
            constr_fn(
                torch.as_tensor(slice_states, dtype=torch.float32, device=DEVICE)
            )
            .reshape(n0, n1)
            .cpu()
            .numpy()
        )

    # ------------------------------------------------------------------
    # Ground-truth value function (optional): same 2-D slice extracted from
    # the cached oracle values if available.  Used to draw a yellow contour
    # of {V_true = 0} on top of every panel.
    # ------------------------------------------------------------------
    gt_slice = None
    truth_dir = f"{WORKSPACE}/data/true_values/{system}"
    if os.path.isfile(f"{truth_dir}/values.npy"):
        values_true = np.load(f"{truth_dir}/values.npy")
        grid_true = np.load(f"{truth_dir}/grid.npy")
        gt_slice = _slice_ground_truth(
            values_true, grid_true, slice_axes, slice_fixed_values,
        )

    # ------------------------------------------------------------------
    # Run each method and compute the tail-min of c(x_k) per trajectory.
    # ------------------------------------------------------------------
    results = []
    for spec in methods:
        traj = _run_method_trajs(spec, dynamics, constr_fn, x0)  # (N, K+1, nx)
        with torch.no_grad():
            c_t = (
                constr_fn(
                    torch.as_tensor(traj.reshape(-1, dynamics.nx), device=DEVICE)
                )
                .reshape(traj.shape[0], traj.shape[1])
                .cpu()
                .numpy()
            )
        results.append(
            {"title": spec["title"], "traj": traj, "tail_min": _tail_min(c_t)}
        )

    # Shared colour scale for the initial-marker fill across all panels.
    init_vals = np.concatenate([r["tail_min"][:, 0] for r in results])
    vmin, vmax = float(init_vals.min()), float(init_vals.max())

    # ------------------------------------------------------------------
    # Plot.
    # ------------------------------------------------------------------
    n_methods = len(results)
    if figsize is None:
        figsize = (5 * n_methods, 5)
    fig, axes = plt.subplots(
        1, n_methods, figsize=figsize, sharey=True, constrained_layout=True,
    )
    if n_methods == 1:
        axes = [axes]

    sc = None
    for ax, r in zip(axes, results):
        ax.set_title(r["title"])
        ax.set_xlabel(slice_axes_labels[0], fontsize=14)
        ax.set_xlim(x_min[ax0], x_max[ax0])
        ax.set_ylim(x_min[ax1], x_max[ax1])

        # Constraint boundary only — clean white background.
        ax.contour(
            g0, g1, constr_grid, levels=[0], colors="black", linewidths=2.0,
        )
        if gt_slice is not None:
            tg0, tg1, values_true_2d = gt_slice
            ax.contour(
                tg0, tg1, values_true_2d, levels=[0],
                colors="red", linewidths=1.5,
            )

        traj = r["traj"]
        tail = r["tail_min"]
        n = traj.shape[0]

        # Connecting lines first so markers sit on top.
        for i in range(n):
            ax.plot(
                traj[i, :, ax0], traj[i, :, ax1],
                color="gray", lw=0.7, alpha=0.5, zorder=1,
            )

        # Trajectory (non-initial) states: small semi-transparent black dots.
        ax.scatter(
            traj[:, 1:, ax0].ravel(), traj[:, 1:, ax1].ravel(),
            color="black", s=12, alpha=0.5, linewidths=0, zorder=2,
        )

        # Initial states: large filled circles coloured by tail-min c(x).
        sc = ax.scatter(
            traj[:, 0, ax0], traj[:, 0, ax1],
            c=tail[:, 0], cmap="viridis", vmin=vmin, vmax=vmax,
            s=60, linewidths=0, zorder=3,
        )

    axes[0].set_ylabel(slice_axes_labels[1], fontsize=14)

    handles = [Line2D([0], [0], color="black", lw=2.0, label="Constraint")]
    if gt_slice is not None:
        handles.append(
            Line2D([0], [0], color="red", lw=1.5, label="Ground truth")
        )
    axes[0].legend(handles=handles, loc="best", fontsize=10)

    cbar = fig.colorbar(sc, ax=axes, location="right", shrink=0.95, pad=0.04, anchor=(0.0, 0.0))
    cbar.ax.set_title(r"$\min_{k \geq 0} c(x_k)$", pad=16)

    if save:
        fig.savefig(
            f"figures/compare/trajectories_{system}.pdf",
            bbox_inches="tight",
            dpi=SAVE_DPI,
        )


In [ ]:
compare_trajectories(
    system="inverted_pendulum",
    methods=TRAJ_METHODS,
    slice_axes=(0, 1),
    slice_axes_labels=(r"$\theta$", r"$\omega$"),
    n_init=200,
    seed=1,
    figsize=(12, 5)
)
